
$\text{Master Cleaning , Combining and Merging Script (Single File)}$
------------------------------------


In [1]:
import os
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# ── ADD-ON (03-08-2026): normalise raw export headers ────────────────────────
# From the 01-08-2026 export onward the app appends form-field MARKERS to the
# header of every special field:  ' *' (required), ' #' (conditional), or both
# (' * #').  e.g. 'Assessment date' -> 'Assessment date *'
#                 "Baby's weight (in kgs)" -> "Baby's weight (in kgs) * #"
# 153 columns across 7 of the 9 exports carry a marker (102 '*', 42 '* #', 9
# '#'); the 22-07 export had none. Every cleaning / combining / merging step
# below looks the columns up by their unmarked name, so stage 1 failed with
# KeyError: ['Assessment date'] and stage 2 with "Baby's weight (in kgs)_2_CG".
# The marker is stripped HERE, at the single point where raw files enter the
# pipeline, so every downstream step sees exactly the names it always saw and no
# other cell needs to change. Verified on the 01-08 export: stripping produces
# no duplicate column names in any file. Exports without asterisks are untouched.
# Set STRIP_REQUIRED_FIELD_MARKER = False to restore the previous behaviour.
STRIP_REQUIRED_FIELD_MARKER = True

import re as _re_hdr

_pd_read_csv_orig = pd.read_csv


def _strip_required_marker(name):
    """Drop trailing form-field markers, whitespace-anchored so only a marker
    that FOLLOWS a space is removed (a '#' inside a real name is never touched):
        'Assessment date *'            -> 'Assessment date'
        "Baby's weight (in kgs) * #"   -> "Baby's weight (in kgs)"
        'Upload picture of weight taken #' -> 'Upload picture of weight taken'
    """
    if not isinstance(name, str):
        return name
    return _re_hdr.sub(r"\s+[*#][\s*#]*$", "", name).strip()


def _read_csv_normalised(*args, **kwargs):
    _df = _pd_read_csv_orig(*args, **kwargs)
    try:
        if STRIP_REQUIRED_FIELD_MARKER and getattr(_df, "columns", None) is not None:
            _new = [_strip_required_marker(c) for c in _df.columns]
            if list(_df.columns) != _new:
                _df.columns = _new
    except Exception:
        pass          # never let header tidying break a read
    return _df


pd.read_csv = _read_csv_normalised
print("Header normalisation active: trailing form-field markers (' *', ' #', ' * #') stripped on read")

## Cleaning process

### Antenatal

In [2]:
def clean_antenatal(input_folder, output_folder, summary_file, run_date):
    """
    Cleans all Antenatal Form datasets for multiple districts.

    Operations:
        • Standardizes all columns containing "Date"
        • Removes duplicates (mutually-exclusive, priority based)
        • Removes rows with missing Assessment Date
        • Saves cleaned CSV for each district
        • Creates an Excel summary with one sheet per district

    Args:
        input_folder  (str): Folder containing raw MP_*.csv files
        output_folder (str): Folder to save cleaned files
        summary_file  (str): Path of Excel summary file
        run_date      (str): Suffix for cleaned filenames (e.g., "2025-11-12")
    """

    os.makedirs(output_folder, exist_ok=True)
    summary_dict = {}

    def remove_duplicates(df, condition_cols, label, before_count, original_count):
        dup_mask = df.duplicated(subset=condition_cols, keep="first")
        removed = dup_mask.sum()
        df = df[~dup_mask]
        percentage = round((removed / original_count) * 100, 2)
        summary_row = [label, removed, percentage, before_count, len(df)]
        return df, len(df), summary_row

    # Process all district files
    for file in os.listdir(input_folder):
        if not file.lower().endswith(".csv") or "antenatal" not in file.lower():
            continue
        
        df = pd.read_csv(os.path.join(input_folder, file))
        
        # Better way to extract district
        parts = file.split("_")
        district = " ".join(parts[1:-3])        # 'East Garo Hills'
    
        original_count = len(df)
        before_count = original_count
        summary_rows = []

        print("\n" + "="*100)
        print(f"🔹 Processing Antenatal file -> {file}")
        print("="*100)

        # ---------------------------------------------
        # 1) Convert all Date columns
        # ---------------------------------------------
        date_cols = [c for c in df.columns if "date" in c.lower()]
        for col in date_cols:
            df[col] = pd.to_datetime(df[col], errors="coerce", format="%d/%m/%Y").dt.date

        # ---------------------------------------------
        # 2) Mutually-exclusive duplicate removal
        # ---------------------------------------------
        if "Submission_ID" in df.columns:
            df, before_count, row = remove_duplicates(
                df, ["Submission_ID"],
                "Duplicates by Submission ID", before_count, original_count)
            summary_rows.append(row)

        if all(c in df.columns for c in ["Case ID", "Assessment date"]):
            df, before_count, row = remove_duplicates(
                df, ["Case ID", "Assessment date"],
                "Duplicates by Case ID + Assessment Date", before_count, original_count)
            summary_rows.append(row)

        if all(c in df.columns for c in ["Case ID", "Assessment date", "Submission_Date"]):
            df, before_count, row = remove_duplicates(
                df, ["Case ID", "Assessment date", "Submission_Date"],
                "Duplicates by Case ID + Assessment Date + Submission Date",
                before_count, original_count)
            summary_rows.append(row)

        # ---------------------------------------------
        # 3) Remove missing Assessment Date
        # ---------------------------------------------
        if "Assessment date" in df.columns:
            removed = df["Assessment date"].isna().sum()
            df = df[df["Assessment date"].notna()]
            percentage = round((removed / original_count) * 100, 2)
            summary_rows.append([
                "Missing_AD", removed, percentage, before_count, len(df)
            ])
            before_count = len(df)

        # ---------------------------------------------
        # 4) Save cleaned dataset
        # ---------------------------------------------
        cleaned_name = f"antenatal_clean_{district}_{run_date}.csv"
        df.to_csv(os.path.join(output_folder, cleaned_name), index=False)
        print(f"Cleaned dataset saved: {cleaned_name}")

        # ---------------------------------------------
        # 5) Add summary sheet for this district
        # ---------------------------------------------
        summary_df = pd.DataFrame(summary_rows, columns=[
            "Cleaning Step", "Rows Removed", "Percentage",
            "Before Count", "After Count"
        ])
        summary_dict[district] = summary_df
        print(f"Summary recorded for district: {district}")

    # ---------------------------------------------
    # Export summary Excel
    # ---------------------------------------------
    with pd.ExcelWriter(summary_file) as writer:
        for district, sm_df in summary_dict.items():
            sm_df.to_excel(writer, sheet_name=(district[:31] or "Sheet1"), index=False)

    print("\n" + "="*100)
    print("Antenatal cleaning completed successfully!")
    print(f"Summary saved: {summary_file}")
    print(f"Cleaned files are stored in: {output_folder}")
    print("="*100 + "\n")


### Case Measurement

In [3]:
def clean_cm(input_folder, output_folder, summary_file, run_date):
    """
    Cleans Case-Measurement (CM) datasets for multiple districts.

    Cleaning operations:
        • Converts all date columns
        • Handles duplicates with mutually-exclusive logic:
            - First remove one copy of exact duplicates
            - Then remove remaining mismatch duplicates
            - Then final de-duplication by Child ID + Visit Date
        • Removes rows missing Visit Date
        • Removes rows where Child ID OR Visit Date is NaN
        • Saves cleaned CSV per district
        • Generates Excel summary with one sheet per district

    Args:
        input_folder  (str): Folder containing raw CM CSVs
        output_folder (str): Folder to save cleaned CM CSVs
        summary_file  (str): Output summary Excel file
        run_date      (str): Suffix for cleaned filenames (e.g., "2025-11-12")
    """

    os.makedirs(output_folder, exist_ok=True)
    summary_dict = {}

    def record_step(step, removed, original_count, before_count, after_count):
        pct = round((removed / original_count) * 100, 2) if removed > 0 else 0
        return [step, removed, pct, before_count, after_count]

    for file in os.listdir(input_folder):
        if not file.endswith(".csv"):
            continue
        if not ("MEASURE" in file.upper()):
            continue

        df = pd.read_csv(os.path.join(input_folder, file))
        # Better way to extract district
        parts = file.split("_")
        district = " ".join(parts[1:-3])        # 'East Garo Hills'

        original_count = len(df)
        before = original_count
        summary_rows = []

        print("\n" + "="*110)
        print(f" Processing CM dataset -> {file}")
        print("="*110)

        # -----------------------------------------
        # 1) Convert date columns
        # -----------------------------------------
        date_cols = [c for c in df.columns if "date" in c.lower()]
        for col in date_cols:
            df[col] = pd.to_datetime(df[col], errors="coerce", format="%d/%m/%Y").dt.date

        # -----------------------------------------
        # 2) Duplicate handling logic
        # -----------------------------------------
        # Detect duplicates by Child ID + Visit Date
        duplicates = df.duplicated(subset=['Child ID', 'Visit Date'], keep=False)
        dups = df[duplicates]

        # A) Exact duplicates
        exact_dup_entries = dups[dups.duplicated(keep=False)].drop_duplicates()
        removed = len(exact_dup_entries)
        df = df.loc[~df.index.isin(exact_dup_entries.index)]
        summary_rows.append(record_step("Exact duplicates - drop one copy",
                                        removed, original_count, before, len(df)))
        before = len(df)

        # B) Remaining mismatched duplicates - drop all
        remaining_dups = dups[~dups.index.isin(exact_dup_entries.index)]
        removed = len(remaining_dups)
        df = df.loc[~df.index.isin(remaining_dups.index)]
        summary_rows.append(record_step("Remaining duplicates (mismatched or multiple)",
                                        removed, original_count, before, len(df)))
        before = len(df)

        # -----------------------------------------
        # 3) Remove non-duplicate rows missing Visit Date
        # -----------------------------------------
        missing_visit = df[df["Visit Date"].isna()]
        removed = len(missing_visit)
        df = df.drop(missing_visit.index)
        summary_rows.append(record_step("Non-duplicate rows missing Visit Date",
                                        removed, original_count, before, len(df)))
        before = len(df)

        # -----------------------------------------
        # 4) Final safety dedupe
        # -----------------------------------------
        before_final = len(df)
        df = df.drop_duplicates(subset=["Child ID", "Visit Date"], keep="last")
        removed = before_final - len(df)
        summary_rows.append(record_step("Final duplicate check (Child ID + Visit Date)",
                                        removed, original_count, before_final, len(df)))
        before = len(df)

        # -----------------------------------------
        # 5) Remove rows with NaN Child ID or Visit Date
        # -----------------------------------------
        nan_rows = df[df[['Child ID', 'Visit Date']].isna().any(axis=1)]
        removed = len(nan_rows)
        df = df.drop(nan_rows.index)
        summary_rows.append(record_step("nan_entries (Child ID OR Visit Date is NaN)",
                                        removed, original_count, before, len(df)))
        before = len(df)

        # -----------------------------------------
        # 6) Save cleaned CSV
        # -----------------------------------------
        cleaned_filename = f"cm_clean_{district}_{run_date}.csv"
        df.to_csv(os.path.join(output_folder, cleaned_filename), index=False)
        print(f"Cleaned file saved -> {cleaned_filename}")

        # -----------------------------------------
        # 7) Store summary
        # -----------------------------------------
        summary_df = pd.DataFrame(
            summary_rows,
            columns=["Cleaning Step", "Rows Removed", "Percentage", "Before Count", "After Count"]
        )
        summary_dict[district] = summary_df
        print(f"Summary recorded for {district}")

    # -----------------------------------------
    # 8) Export summary Excel
    # -----------------------------------------
    with pd.ExcelWriter(summary_file) as writer:
        for district, sm in summary_dict.items():
            sm.to_excel(writer, sheet_name=(district[:31] or "Sheet1"), index=False)

    print("\n" + "="*110)
    print("Case-Measurement cleaning completed successfully for ALL districts!")
    print(f"Summary File -> {summary_file}")
    print(f"Cleaned CSVs stored in -> {output_folder}")
    print("="*110 + "\n")


### BF

In [4]:
def clean_bf(input_folder, output_folder, summary_file, run_date):
    """
    Breastfeeding (BF) Form Cleaning Script
    Description:
        Cleans Breastfeeding form datasets for multiple districts automatically.

        Outputs:
            • Cleaned CSV for each district
            • One Excel summary (one sheet per district)
    """
    
    os.makedirs(output_folder, exist_ok=True)
    summary_dict = {}

    # ---------------------------------------------------------
    # Log function
    # ---------------------------------------------------------
    def record_step(step, removed, original_count, before_count, after_count):
        percentage = round((removed / original_count) * 100, 2) if removed > 0 else 0
        print(f"{step}: {removed} removed ({percentage}%)")
        return [step, removed, percentage, before_count, after_count]

    # ---------------------------------------------------------
    # MAIN LOOP
    # ---------------------------------------------------------
    for file in os.listdir(input_folder):
        if not file.endswith(".csv") or "BF" not in file:
            continue

        print("\n" + "="*110)
        print(f"Processing BF dataset -> {file}")
        print("="*110)

        filepath = os.path.join(input_folder, file)
        df = pd.read_csv(filepath)
                # Better way to extract district
        parts = file.split("_")
        district = " ".join(parts[1:-3])        # 'East Garo Hills'
        original_count = len(df)
        before = original_count
        summary_rows = []

        # Convert date columns
        date_cols = [c for c in df.columns if "date" in c.lower()]
        for col in date_cols:
            df[col] = pd.to_datetime(df[col], errors='coerce', format="%d/%m/%Y").dt.date

        # Identify duplicate groups
        Dup = df[df.duplicated(subset=['Assessment date','Submission_Date','Case ID'], keep=False)]
        dups = df[df.duplicated(subset=['Case ID','Assessment date'], keep=False)]

        # Duplicated rows with NaN date/Submission_ID
        Nan_entries = df.loc[
            dups.index.intersection(
                df[df[['Assessment date','Submission_ID']].isna().any(axis=1)].index
            )
        ]

        # Remove Dup
        removed_dup = len(Dup)
        df = df.iloc[~df.index.isin(Dup.index)]
        summary_rows.append(record_step(
            "Duplicates (Assessment date + Submission_Date + Case ID)",
            removed_dup, original_count, before, len(df)
        ))
        before = len(df)

        # Remove Nan_entries
        removed_nan = len(Nan_entries)
        df = df.iloc[~df.index.isin(Nan_entries.index)]
        summary_rows.append(record_step(
            "Duplicates with NaN (Assessment date / Submission_ID)",
            removed_nan, original_count, before + removed_nan, len(df)
        ))
        before = len(df)

        # Remove missing Assessment_date
        removed_nan_assess = df['Assessment date'].isna().sum()
        df = df[df['Assessment date'].notna()]
        summary_rows.append(record_step(
            "Rows with missing Assessment date",
            removed_nan_assess, original_count, before, len(df)
        ))
        before = len(df)

        # Remove rows where all other columns empty
        key_cols = ['Case ID', 'Assessment date', 'Submission_Date', 'Submission_ID']
        key_cols = [c for c in key_cols if c in df.columns]
        other_cols = [c for c in df.columns if c not in key_cols]

        if other_cols:
            empty_mask = df[other_cols].apply(lambda row: all(
                (pd.isna(x) or str(x).strip() == "") for x in row), axis=1)
            empty_other_columns = df[empty_mask]
        else:
            empty_other_columns = df.iloc[0:0]

        removed_empty = len(empty_other_columns)
        df = df.iloc[~df.index.isin(empty_other_columns.index)]
        summary_rows.append(record_step(
            "Rows with all non-key columns empty (removed completely)",
            removed_empty, original_count, before, len(df)
        ))
        before = len(df)

        # Deduplicate Case ID + Assessment_date keep last
        before_dedup = len(df)
        df = df.drop_duplicates(subset=['Case ID', 'Assessment date'], keep='last')
        removed_case_assess = before_dedup - len(df)
        summary_rows.append(record_step(
            "Drop remaining duplicates (Case ID + Assessment date)",
            removed_case_assess, original_count, before_dedup, len(df)
        ))

        # Save cleaned dataset
        cleaned_filename = f"bf_clean_{district}_{run_date}.csv"
        df.to_csv(os.path.join(output_folder, cleaned_filename), index=False)
        print(f"Cleaned dataset saved -> {cleaned_filename}")

        # Store summary
        summary_df = pd.DataFrame(summary_rows,
                                  columns=["Cleaning Step", "Rows Removed", "Percentage",
                                           "Before Count", "After Count"])
        summary_dict[district] = summary_df
        print(f"Summary stored for -> {district}")

    # Export final summary Excel
    with pd.ExcelWriter(summary_file) as writer:
        for district, sm in summary_dict.items():
            sm.to_excel(writer, sheet_name=(district[:31] or "Sheet1"), index=False)

    print("\n" + "="*110)
    print("Breastfeeding form cleaning completed successfully for ALL districts!")
    print(f"Summary File -> {summary_file}")
    print(f"Cleaned CSVs stored in -> {output_folder}")
    print("="*110 + "\n")


### CF

In [5]:
def clean_cf(input_folder, output_folder, summary_file, run_date):
    """
    Cleaning function for Case-Food (CF) forms.
    Steps followed:
        1. Convert all date-like columns to datetime
        2. Remove rows with missing Assessment_date
        3. Detect duplicates (Assessment_date + Case ID)
        4. Remove duplicates -> keep last
        5. Save cleaned CSV per district
        6. Create Excel summary with notebook-style extra sheets
    """
    os.makedirs(output_folder, exist_ok=True)
    summary_dict = {}

    def record_step(step, removed, original, before, after):
        pct = round((removed / original) * 100, 2) if removed else 0
        print(f"{step}: Removed {removed} ({pct}%)")
        return [step, removed, pct, before, after]

    # ------------------------- MAIN LOOP -------------------------
    for file in os.listdir(input_folder):
        if not file.endswith(".csv") or "CF" not in file.upper():
            continue

        print("\n" + "="*120)
        print(f"Processing CF Dataset -> {file}")
        print("="*120)

        df = pd.read_csv(os.path.join(input_folder, file))
                # Better way to extract district
        parts = file.split("_")
        district = " ".join(parts[1:-3])        # 'East Garo Hills'
        original_count = len(df)
        before = original_count
        summary_rows = []

        # 1) Convert all date-like columns
        date_cols = [c for c in df.columns if "date" in c.lower()]
        for col in date_cols:
            df[col] = pd.to_datetime(df[col], errors="coerce", format="%d/%m/%Y")

        # 2) Missing Assessment_date
        missing_assess = df[df["Assessment date"].isna()]
        removed = len(missing_assess)
        df = df.dropna(subset=["Assessment date"])
        summary_rows.append(record_step("Missing Assessment date", removed, original_count, before, len(df)))
        before = len(df)

        # 3) Duplicate groups
        duplicate_groups = df[df.duplicated(subset=["Assessment date", "Case ID"], keep=False)]

        # 4) Drop duplicates keep last
        before_dedup = len(df)
        df = df.drop_duplicates(subset=["Assessment date", "Case ID"], keep="last")
        removed = before_dedup - len(df)
        summary_rows.append(record_step("Dropped duplicates (Assessment date + Case ID)",
                                        removed, original_count, before_dedup, len(df)))

        # 5) Save cleaned dataset
        cleaned_filename = f"cf_clean_{district}_{run_date}.csv"
        df.to_csv(os.path.join(output_folder, cleaned_filename), index=False)
        print(f"Cleaned file saved -> {cleaned_filename}")

        # 6) Summary + extra notebook sheets
        summary_df = pd.DataFrame(summary_rows,
                                  columns=["Cleaning Step", "Rows Removed", "Percentage",
                                           "Before Count", "After Count"])
        extras = {
            "Missing_AssD": missing_assess,
            "Duplicate Groups": duplicate_groups
        }
        summary_dict[district] = (summary_df, extras)

        print(f"Summary stored for {district}")

    # 7) Export multi-sheet summary Excel
    with pd.ExcelWriter(summary_file, engine='openpyxl') as writer:
        for district, (summary_df, extras) in summary_dict.items():
            
            # Safe Summary sheet
            sheet_name = f"{district}_Summary"[:31]
            summary_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            for sheetname, df_sheet in extras.items():
                full_name = f"{district}_{sheetname}"
                safe_name = full_name[:28] + "..." if len(full_name) > 31 else full_name
                df_sheet.to_excel(writer, sheet_name=safe_name, index=False)

    print("\n" + "="*120)
    print("Case-Food (CF) cleaning completed successfully!")
    print(f"Summary Excel -> {summary_file}")
    print(f"Cleaned CSVs saved in -> {output_folder}")
    print("="*120 + "\n")


### Check Growth

In [6]:
def clean_cg(input_folder,
             output_folder,
             summary_file,run_date):
    """
    Cleaning function for Check Growth (CG) forms.
    Steps followed:
        • Convert all date columns
        • Drop rows missing Measurement_date
        • Drop duplicates on (Submission_ID, Case ID) -> keep='last'
        • Save cleaned CSV per district
        • Generate final summary Excel (district-wise)
    """
    os.makedirs(output_folder, exist_ok=True)
    summary_dict = {}

    def record_step(step, removed, original_count, before_count, after_count):
        pct = round((removed / original_count) * 100, 2) if removed else 0
        print(f"{step}: {removed} removed ({pct}%)")
        return [step, removed, pct, before_count, after_count]

    # ------------------- MAIN LOOP -------------------
    for file in os.listdir(input_folder):

        # Identify CG dataset
        if not file.lower().endswith(".csv"):
            continue
        if not ("check_growth" in file.lower() or "cg" in file.lower()):
            continue

        print("\n" + "="*120)
        print(f"Processing Check Growth dataset -> {file}")
        print("="*120)

        df = pd.read_csv(os.path.join(input_folder, file))
                # Better way to extract district
        parts = file.split("_")
        district = " ".join(parts[1:-3])        # 'East Garo Hills'
        original_count = len(df)
        before = original_count
        summary_rows = []

        # 1) Convert all date columns
        date_cols = [c for c in df.columns if "date" in c.lower()]
        for col in date_cols:
            df[col] = pd.to_datetime(df[col], errors="coerce",
                                     format="%d/%m/%Y").dt.date

        # 2) Duplicate removal: (Submission_ID, Case ID) - keep last
        if ("Submission_ID" in df.columns) and ("Case ID" in df.columns):
            before_dup = len(df)
            df = df.drop_duplicates(subset=["Submission_ID", "Case ID"], keep="last")
            removed = before_dup - len(df)
            summary_rows.append(
                record_step("Duplicates on (Submission_ID, Case ID) - keep last",
                            removed, original_count, before_dup, len(df))
            )
            before = len(df)

        # 3) Drop missing Measurement date
        missing_measure = df["Measurement date"].isna()
        removed = missing_measure.sum()
        df = df[~missing_measure]
        summary_rows.append(
            record_step("Missing Measurement date removed",
                        removed, original_count, before, len(df))
        )
        before = len(df)

        # 4) Save cleaned CSV
        cleaned_filename = f"cg_clean_{district}_{run_date}.csv"
        df.to_csv(os.path.join(output_folder, cleaned_filename), index=False)
        print(f"Cleaned CSV saved -> {cleaned_filename}")

        # 5) Add summary entry for Excel
        summary_df = pd.DataFrame(
            summary_rows,
            columns=["Cleaning Step", "Rows Removed", "Percentage",
                     "Before Count", "After Count"]
        )
        summary_dict[district] = summary_df
        print(f"Summary saved for -> {district}")

    # 6) Export summary Excel
    with pd.ExcelWriter(summary_file) as writer:
        for district, summary_df in summary_dict.items():
            summary_df.to_excel(writer, sheet_name=(district[:31] or "Sheet1"), index=False)

    print("\n" + "="*120)
    print("Check Growth (CG) cleaning completed successfully!")
    print(f"Summary Excel saved -> {summary_file}")
    print(f"Cleaned CSVs saved in -> {output_folder}")
    print("="*120 + "\n")


### Child

In [7]:
def clean_child(input_folder, output_folder, summary_file, run_date):
    """
    CHILD FORM CLEANING FUNCTION

    Rules applied:
    1. Convert all columns containing "date" to datetime using format "%d/%m/%Y"
    2. Drop duplicates on Case ID -> keep last
    3. Display duplicates on Submission_ID
    4. Drop duplicates on Submission_ID -> keep last
    5. Save cleaned outputs + summary for counts removed
    """
    os.makedirs(output_folder, exist_ok=True)
    summary_dict = {}

    def record_step(step, removed, original, before, after):
        pct = round((removed / original) * 100, 2) if original else 0
        print(f"{step}: {removed} removed ({pct}%)")
        return [step, removed, pct, before, after]

    # --------------------------------------
    # PROCESS EACH CHILD FILE
    # --------------------------------------
    for file in os.listdir(input_folder):
        if not file.lower().endswith(".csv"):
            continue
        if "child" not in file.lower():       # ensures only Child files
            continue

        print("\n" + "="*110)
        print(f"Processing -> {file}")
        print("="*110)

        path = os.path.join(input_folder, file)
        df = pd.read_csv(path)
                # Better way to extract district
        parts = file.split("_")
        district = " ".join(parts[1:-3])        # 'East Garo Hills'

        original_count = len(df)
        summary_rows = []

        # 1) Convert date columns
        date_cols = [c for c in df.columns if "date" in c.lower()]
        for col in date_cols:
            df[col] = pd.to_datetime(df[col], errors="coerce", format="%d/%m/%Y")

        # 2) Remove duplicates on Case ID (keep last)
        before_case = len(df)
        df.drop_duplicates(subset="Case ID", keep="last", inplace=True)
        removed = before_case - len(df)
        summary_rows.append(
            record_step("Duplicates on Case ID removed (keep last)",
                        removed, original_count, before_case, len(df))
        )

        # 3) Display duplicates on Submission_ID (diagnostic only)
        if "Submission_ID" in df.columns:
            dup_display = df[df.duplicated(subset="Submission_ID", keep=False)]
            if len(dup_display) > 0:
                print("\n Duplicate Submission_ID rows found:")
                print(dup_display)

        # 4) Remove duplicates on Submission_ID (keep last)
        if "Submission_ID" in df.columns:
            before_sub = len(df)
            df.drop_duplicates(subset="Submission_ID", keep="last", inplace=True)
            removed = before_sub - len(df)
            summary_rows.append(
                record_step("Duplicates on Submission_ID removed (keep last)",
                            removed, original_count, before_sub, len(df))
            )

        # 5) Save cleaned output
        outname = f"child_clean_{district}_{run_date}.csv"
        df.to_csv(os.path.join(output_folder, outname), index=False)
        print(f"Saved cleaned file -> {outname}")

        summary_dict[district] = pd.DataFrame(
            summary_rows, columns=["Cleaning Step", "Removed", "Percent", "Before", "After"]
        )

    # Export summary summary
    with pd.ExcelWriter(summary_file) as writer:
        for district, sm in summary_dict.items():
            sm.to_excel(writer, sheet_name=(district[:31] or "Sheet1"), index=False)

    print("\n" + "="*110)
    print("Child forms cleaned successfully!")
    print(f"Summary saved -> {summary_file}")
    print("="*110)


### Case Report

In [8]:
import os
import pandas as pd

def clean_cr(input_folder, output_folder, run_date, summary_file):
    """
    Case-Report cleaning function
    ------------------------------------------------
    • Cleans all Case Report CSVs
    • Saves cleaned district-wise CSVs
    • Produces ONE summary sheet per district
    """

    os.makedirs(output_folder, exist_ok=True)

    def log_step(step, removed, before, after):
        pct = round((removed / before) * 100, 2) if before else 0
        return [step, removed, pct, before, after]

    # --------------------
    # Pregnancy stage logic
    # --------------------
    post_delivery_stages = [
        'Postnatal Care (Child is 5 months of age or younger)',
        'Postnatal Care (Child is older than 5 months of age)',
        'At the time of delivery'
    ]

    pre_del_list = [
        'First Trimester', 'Second Trimester',
        'Third Trimester', 'Pre-pregnancy'
    ]

    def categorize(stage):
        if pd.isna(stage):
            return 'unknown'
        if stage in pre_del_list:
            return 'pre_del'
        if stage in post_delivery_stages:
            return 'post_del'
        return 'unknown'

    summary_dict = {}

    # ==========================
    # Process files
    # ==========================
    for file in os.listdir(input_folder):
        if not file.lower().endswith(".csv"):
            continue
        if "case_report" not in file.lower():
            continue

        print(f"\nProcessing: {file}")

        df = pd.read_csv(os.path.join(input_folder, file), dtype=object)
        before = len(df)
        summary_rows = []

        # --------------------
        # Convert date columns
        # --------------------
        date_cols = [c for c in df.columns if "date" in c.lower()]
        for c in date_cols:
            df[c] = pd.to_datetime(df[c], errors="coerce", format="%d/%m/%Y")

        # --------------------
        # Pregnancy Stage check
        # --------------------
        if "Pregnancy Stage" not in df.columns:
            print("Skipped – Pregnancy Stage column missing")
            continue

        # --------------------
        # Missing Pregnancy Stage
        # --------------------
        removed = df["Pregnancy Stage"].isna().sum()
        df = df.dropna(subset=["Pregnancy Stage"])
        summary_rows.append(log_step("Missing Pregnancy Stage", removed, before, len(df)))
        before = len(df)

        # --------------------
        # Categorize
        # --------------------
        df["Category"] = df["Pregnancy Stage"].apply(categorize)

        # --------------------
        # Remove IIT HST accounts
        # --------------------
        if "User Role" in df.columns:
            removed = (df["User Role"] == "IIT HST Trainer + Special Testing Account").sum()
            df = df[df["User Role"] != "IIT HST Trainer + Special Testing Account"]
            summary_rows.append(
                log_step(
                    "IIT HST Trainer + Special Testing Account cases",
                    removed, before, len(df)
                )
            )
            before = len(df)

        if "User Role_CR" in df.columns:
            removed = (df["User Role_CR"] == "IIT HST Trainers").sum()
            df = df[df["User Role_CR"] != "IIT HST Trainers"]
            summary_rows.append(
                log_step(
                    "IIT HST Trainers cases",
                    removed, before, len(df)
                )
            )
            before = len(df)

        # --------------------
        # Missing Child ID (post-delivery only)
        # --------------------
        if "Child ID" in df.columns:
            mask = df["Child ID"].isna() & df["Pregnancy Stage"].isin(post_delivery_stages)
            removed = mask.sum()
            df = df[~mask]
            summary_rows.append(
                log_step("Missing Child ID for Post Delivery", removed, before, len(df))
            )
            before = len(df)

        # --------------------
        # Duplicate removal
        # --------------------
        dup_key = ["Mother Adoption date", "Case ID"]

        pre_df = df[df["Category"] == "pre_del"]
        post_df = df[df["Category"] == "post_del"]

        if all(c in pre_df.columns for c in dup_key):
            removed = pre_df.duplicated(subset=dup_key).sum()
            pre_df = pre_df.drop_duplicates(subset=dup_key)
        else:
            removed = 0

        summary_rows.append(
            log_step("Duplicate PRE (Mother Adoption date + Case ID)", removed, before, before - removed)
        )

        if all(c in post_df.columns for c in dup_key):
            removed = post_df.duplicated(subset=dup_key).sum()
            post_df = post_df.drop_duplicates(subset=dup_key)
        else:
            removed = 0

        summary_rows.append(
            log_step("Duplicate POST (Mother Adoption date + Case ID)", removed, before, before - removed)
        )

        # --------------------
        # Combine & save cleaned
        # --------------------
        cleaned_df = pd.concat([pre_df, post_df], ignore_index=True)

                # Better way to extract district
        parts = file.split("_")
        district = " ".join(parts[1:-3])        # 'East Garo Hills'
        cleaned_df.to_csv(
            os.path.join(output_folder, f"cr_clean_{district}_{run_date}.csv"),
            index=False
        )

        # --------------------
        # Store summary ONLY
        # --------------------
        summary_dict[district] = pd.DataFrame(
            summary_rows,
            columns=[
                "Cleaning Step",
                "Rows Removed",
                "Percentage",
                "Before Count",
                "After Count"
            ]
        )

    # ==========================
    # Write summary Excel ONLY
    # ==========================
    with pd.ExcelWriter(summary_file, engine="openpyxl") as writer:
        for district, summary_df in summary_dict.items():
            summary_df.to_excel(writer, sheet_name=(district[:31] or "Sheet1"), index=False)

    print("\n✅ clean_cr() completed – summary saved at:", summary_file)


### Mother

In [9]:
def clean_mother(input_folder, output_folder, run_date, summary_file):
    """
    Cleans Mother Form datasets for all districts.

    Steps:
        1. Convert all columns containing 'date' -> datetime
        2. Drop duplicates on Case ID (keep last)
        3. Report duplicate Submission_ID groups
        4. Drop duplicates on Submission_ID (keep last)
        5. Report duplicate (Case ID + Mother's_Name)
        6. Drop duplicates on (Case ID + Mother's_Name)
        7. Drop rows with missing Adoption_date
        8. Export cleaned CSV per district
        9. Export multi-sheet summary Excel

    Parameters:
        input_folder  : Folder path containing raw CSVs
        output_folder : Folder path to save cleaned CSVs
        user_date     : Date string for output filenames (YYYY-MM-DD)
        summary_path  : Full path for summary Excel
    """
    os.makedirs(output_folder, exist_ok=True)
    summary_dict = {}     # store district -> summary

    def log(step, removed, before, after):
        pct = round((removed / before) * 100, 2) if before else 0
        return [step, removed, pct, before, after]

    print("\n Starting Mother File Cleaning")

    for file in os.listdir(input_folder):
        if not file.endswith(".csv") or "MOTHER" not in file.upper():
            continue

        # ---- FIX (04-08-2026): the Protein Intake export is NOT a Mother file ----
        # "MP_<District>_Mother's_Protein_Intake_<date>.csv".upper() contains "MOTHER",
        # so the Protein form used to be cleaned here as well. Because the district
        # parser gave it a different output name it did not overwrite the real mother
        # file; both were concatenated in step 2, and the 1596 empty protein shells
        # (which sort first) then won drop_duplicates(keep="first") during merging and
        # wiped the mother attributes for 1596 of 2004 cases. The Protein form has its
        # own cleaner, clean_protein().
        if "PROTEIN" in file.upper():
            print(f"  [skip] {file} -> Protein Intake form, handled by clean_protein()")
            continue

        print("\n" + "="*110)
        print(f"Processing Mother dataset -> {file}")
        print("="*110)

        df = pd.read_csv(os.path.join(input_folder, file))
                # Better way to extract district
        # ---- FIX (04-08-2026): district parsing was positional and form-specific ----
        # " ".join(parts[1:-3]) assumed the form name is exactly two "_" tokens, so
        # 'MP_Ujjain_Mother_<date>.csv' (4 tokens) produced an EMPTY district while
        # the protein file produced "Ujjain Mother's". Take everything between the
        # state token and the form token instead:
        #   MP_Ujjain_Mother_<date>.csv          -> 'Ujjain'
        #   ML_East_Garo_Hills_Mother_<date>.csv -> 'East Garo Hills'
        parts = file.split("_")
        try:
            _form_i = next(k for k, p in enumerate(parts) if p.upper().startswith("MOTHER"))
        except StopIteration:
            _form_i = max(len(parts) - 3, 1)     # original behaviour as a fallback
        district = " ".join(parts[1:_form_i])   # 'East Garo Hills'
        before = len(df)
        summary_rows = []

        # ------------------- 1) Convert all date columns -------------------
        date_cols = [c for c in df.columns if "date" in c.lower()]
        for c in date_cols:
            df[c] = pd.to_datetime(df[c], errors='coerce', format="%d/%m/%Y")
        summary_rows.append(log("Converted date columns", 0, before, len(df)))

        # ------------------- 1b) Drop Draft / Test submissions -------------------
        # FIX (04-08-2026): the app exports unfinished ("Draft") and dummy ("Test")
        # submissions. reshape_mother() drops the two FLAG COLUMNS but the ROWS used to
        # survive into mother_combined.csv as ordinary data (40 Case IDs in UJ 010826,
        # none of which has a case report).
        for _flag in ["Draft", "Test"]:
            if _flag in df.columns:
                _before_flag = len(df)
                _is_flag = (df[_flag].astype(str).str.strip().str.lower()
                            .isin(["true", "1", "yes"]))
                df = df[~_is_flag]
                summary_rows.append(
                    log(f"Dropped {_flag}=true submissions",
                        _before_flag - len(df), _before_flag, len(df))
                )

        # ------------------- 2) Drop duplicates on Case ID -------------------
        before_case = len(df)
        df = df.drop_duplicates(subset="Case ID", keep="last")
        summary_rows.append(
            log("Removed duplicates on Case ID", before_case - len(df), before_case, len(df))
        )

        # ------------------- 3) Detect duplicate Submission_ID groups -------------------
        dups_sub = df[df.duplicated(subset="Submission_ID", keep=False)]
        summary_rows.append(
            log("Detected duplicate Submission_ID groups", len(dups_sub), len(df), len(df))
        )

        # ------------------- 4) Drop duplicates on Submission_ID -------------------
        before_sub = len(df)
        df = df.drop_duplicates(subset="Submission_ID", keep="last")
        summary_rows.append(
            log("Removed duplicates on Submission_ID", before_sub - len(df), before_sub, len(df))
        )

        # ------------------- 5 & 6) Case ID + Mother's Name -------------------
        # FIX (04-08-2026): the real column is "Mother's Name" (space, not underscore),
        # so this guard was never true and steps 5-6 below have never run on any dataset.
        # (Case ID is already unique after step 2, so enabling them is a no-op in
        # practice -- it only makes the summary sheet report the check honestly.)
        if {"Case ID", "Mother's Name"}.issubset(df.columns):
            dup_case_mom = df[df.duplicated(subset=["Case ID", "Mother's Name"], keep=False)]
            summary_rows.append(
                log("Detected duplicate Case ID + Mother's Name", len(dup_case_mom), len(df), len(df))
            )

            before_pair = len(df)
            df = df.drop_duplicates(subset=["Case ID", "Mother's Name"], keep="last")
            summary_rows.append(
                log("Removed duplicates on Case ID + Mother's Name", before_pair - len(df), before_pair, len(df))
            )

        # ------------------- 7) Drop Missing Adoption_date -------------------
        if "Adoption date" in df.columns:
            before_adopt = len(df)
            df = df.dropna(subset=["Adoption date"])
            summary_rows.append(
                log("Dropped rows with missing Adoption date", before_adopt - len(df), before_adopt, len(df))
            )

        # ------------------- Save cleaned CSV -------------------
        cleaned_filename = f"mother_clean_{district[:31]}_{run_date}.csv"
        df.to_csv(os.path.join(output_folder, cleaned_filename), index=False)
        print(f"Saved: {cleaned_filename}")

        # ------------------- Store summary -------------------
        summary_dict[district] = pd.DataFrame(
            summary_rows,
            columns=["Cleaning Step", "Rows Removed", "Percentage", "Before Count", "After Count"]
        )

    # ------------------- Export Summary Excel -------------------
    with pd.ExcelWriter(summary_file) as writer:
        for district, sm in summary_dict.items():
            sm.to_excel(writer, sheet_name=(district[:31] or "Sheet1"), index=False)

    print("\n Mother File Cleaning Completed Successfully!")
    print(f"Summary saved -> {summary_file}")
    print(f"Cleaned files saved -> {output_folder}")


### Protein

In [10]:
def clean_protein(input_folder, output_folder, summary_file, run_date):
    """
    Simplified Protein Form Cleaning Function
    Performs exactly 4 operations:
        1) Remove Submission_ID duplicates
        2) Remove Assessment date + Case ID duplicates
        3) Drop missing Assessment date rows
        4) Keep only required columns in final output
    """
    os.makedirs(output_folder, exist_ok=True)

    keep_cols = [
        'User Acc ID', 'User Name', 'NGO/Facility', 'Submission_ID', 'Submission_Date',
        'Case ID', 'Assessment date', 'Diet type', "Mother's status",
        'Total protein from all foods (grams)', 'Excellent and high quality protein (grams)',
        'User Reg Id'
    ]

    summary_dict = {}

    for file in os.listdir(input_folder):

        if not file.endswith(".csv"):
            continue
        if "PROTEIN" not in file.upper():
            continue

        print("\n" + "="*110)
        print(f"Processing -> {file}")
        print("="*110)

        path = os.path.join(input_folder, file)
        df = pd.read_csv(path)
                # Better way to extract district
        parts = file.split("_")
        district = " ".join(parts[1:-3])        # 'East Garo Hills'

        original = len(df)
        steps = []

        # Convert date columns
        for col in df.columns:
            if "date" in col.lower():
                df[col] = pd.to_datetime(df[col], errors='coerce', format="%d/%m/%Y").dt.date

        # 1) Remove Submission_ID duplicates
        before = len(df)
        df = df.drop_duplicates(subset=["Submission_ID"], keep="first")
        steps.append(["Submission_ID duplicates removed", before - len(df)])

        # 2) Remove Case ID + Assessment date duplicates
        before = len(df)
        df = df.drop_duplicates(subset=["Case ID", "Assessment date"], keep="last")
        steps.append(["Case ID + Assessment date duplicates removed", before - len(df)])

        # 3) Drop missing Assessment date rows
        before = len(df)
        df = df[df["Assessment date"].notna()]
        steps.append(["Rows with missing Assessment date removed", before - len(df)])

        # 4) Keep required columns
        missing_cols = [c for c in keep_cols if c not in df.columns]
        if missing_cols:
            print(f"Columns missing in {district}: {missing_cols}")
        df = df[[c for c in keep_cols if c in df.columns]]

        # Save cleaned CSV
        output_filename = f"protein_clean_{district}_{run_date}.csv"
        df.to_csv(os.path.join(output_folder, output_filename), index=False)
        print(f"Cleaned file saved -> {output_filename}")

        # Save steps summary
        summary_df = pd.DataFrame(steps, columns=["Step", "Rows Removed"])
        summary_df["Original Rows"] = original
        summary_df["Final Rows"] = len(df)
        summary_dict[district] = summary_df

    # Export Summary Excel
    with pd.ExcelWriter(summary_file) as writer:
        for district, sm in summary_dict.items():
            sm.to_excel(writer, sheet_name=(district[:31] or "Sheet1"), index=False)
    

    print("\n" + "="*110)
    print("Protein cleaning complete for ALL districts!")
    print(f"Summary Excel -> {summary_file}")
    print(f"Cleaned CSVs saved in -> {output_folder}")
    print("="*110 + "\n")


### Running Cleaning script

In [11]:
# FIX (04-08-2026): cleaned/<form>/ was never cleared between runs, and the cleaned
# file name carries the district. A file written by an EARLIER run (different district,
# or a name this run no longer produces -- e.g. the old
# "mother_clean_Ujjain Mother's_<date>.csv" protein shell) was still picked up by
# run_step2_generic() and concatenated into this run's combined file. Set to False to
# restore the previous accumulate-forever behaviour.
CLEAR_STALE_CLEANED_FILES = True


def run_all_cleaning(input_folder, output_folder, summary_folder, run_date):

    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(summary_folder, exist_ok=True)

    cleaning_jobs = [
        ("antenatal", clean_antenatal),
        ("case_measurement", clean_cm),
        ("case_report", clean_cr),
        ("bf", clean_bf),
        ("cf", clean_cf),
        ("child", clean_child),
        ("mother", clean_mother),
        ("protein", clean_protein),
        ("check_growth", clean_cg)
    ]

    print("\n Starting full cleaning pipeline...\n")

    for dataset_name, cleaning_func in cleaning_jobs:

        # detect whether any file belongs to this dataset
        matched = any(dataset_name in f.lower() for f in os.listdir(input_folder))

        if not matched:
            print(f" Skipping {dataset_name} (no files found)")
            continue

        print(f"\n=== Running {dataset_name.upper()} cleaning ===\n")

        dataset_output = os.path.join(output_folder, dataset_name)
        os.makedirs(dataset_output, exist_ok=True)

        # Clear this form's folder ONCE per run, before any file is written. Every
        # district of the CURRENT run is written after this point, so multi-district
        # states (e.g. Meghalaya) still accumulate correctly within a run.
        if CLEAR_STALE_CLEANED_FILES:
            for _old in sorted(os.listdir(dataset_output)):
                if _old.lower().endswith(".csv"):
                    os.remove(os.path.join(dataset_output, _old))
                    print(f"  [stale] removed file left by a previous run: {dataset_name}/{_old}")

        summary_file = os.path.join(summary_folder, f"{dataset_name}_summary.xlsx")

        # your cleaning function runs exactly as before
        cleaning_func(
            input_folder=input_folder,
            output_folder=dataset_output,
            summary_file=summary_file,
            run_date=run_date
        )

    print("\n All dataset types cleaned!")


In [12]:
# ==============================================================================
# DYNAMIC INPUT / OUTPUT NAMING (naming only -- cleaning logic unchanged)
# The raw data folder is named like:  input_folder_1107(CUD)_1707(DD)_Ujjain
#   date before (CUD) = current-upload date  DDMM[YY]
#   date before (DD)  = download date        DDMM[YY]
#   trailing word     = district / state     (Meghalaya->ML, Jalna->JL, Ujjain->UJ)
# and the merged output of this script is named:  UJ_110726_V(170726).csv
# ==============================================================================
import os
import re
from datetime import datetime

DISTRICT_PREFIX = {"MEGHALAYA": "ML", "JALNA": "JL", "UJJAIN": "UJ"}

def resolve_raw_input_folder(parent="input_folder"):
    """Return the folder that actually holds the raw CSVs: the folder injected
    by the master pipeline (PIPELINE_RAW_INPUT_DIR), else the newest dated
    'input_folder_*(CUD)*' subfolder inside `parent`, else `parent` itself
    (the original behaviour)."""
    injected = globals().get('PIPELINE_RAW_INPUT_DIR', None)
    if injected:
        return str(injected)
    if os.path.isdir(parent):
        subs = [os.path.join(parent, d) for d in os.listdir(parent)
                if os.path.isdir(os.path.join(parent, d)) and '(CUD)' in d.upper()]
        if subs:
            return max(subs, key=os.path.getmtime)
    return parent

def derive_output_base_from_folder(folder):
    """input_folder_1107(CUD)_1707(DD)_Ujjain -> UJ_110726_V(170726).
    4-digit dates (DDMM) get the current 2-digit year appended; 6-digit dates
    (DDMMYY) are used as-is. Returns None if the folder name carries no
    (CUD)/(DD) dates."""
    name = os.path.basename(str(folder).rstrip('/\\'))
    m = re.search(r'(\d{4,6})\s*\(CUD\)[_\s]*(\d{4,6})\s*\(DD\)[_\s]*([A-Za-z]+)',
                  name, flags=re.IGNORECASE)
    if not m:
        return None
    yy = datetime.now().strftime('%y')
    cud = m.group(1) if len(m.group(1)) == 6 else m.group(1) + yy
    dd = m.group(2) if len(m.group(2)) == 6 else m.group(2) + yy
    district = m.group(3).strip().upper()
    prefix = DISTRICT_PREFIX.get(district, district[:2].upper())
    return f"{prefix}_{cud}_V({dd})"

RAW_INPUT_FOLDER = resolve_raw_input_folder("input_folder")
MERGED_OUTPUT_BASE = derive_output_base_from_folder(RAW_INPUT_FOLDER)
if MERGED_OUTPUT_BASE is None:
    MERGED_OUTPUT_BASE = "ML_100526_V(120526)"   # original hardcoded fallback
    print(f"WARNING: no (CUD)/(DD) dates in '{RAW_INPUT_FOLDER}' -> falling back to {MERGED_OUTPUT_BASE}")
print(f"Raw input folder  : {RAW_INPUT_FOLDER}")
print(f"Merged output name: {MERGED_OUTPUT_BASE}.csv")

run_all_cleaning(
    input_folder=RAW_INPUT_FOLDER,
    output_folder="cleaned_files",
    summary_folder="summaries",
    run_date="2026-04-05"
)



 Starting full cleaning pipeline...


=== Running ANTENATAL cleaning ===


🔹 Processing Antenatal file -> ML_Eastern_West_Khasi_Hills_Antenatal_care_2026-06-14-132257.csv
Cleaned dataset saved: antenatal_clean_Eastern West Khasi Hills_2026-04-05.csv
Summary recorded for district: Eastern West Khasi Hills

🔹 Processing Antenatal file -> ML_East_Garo_Hills_Antenatal_care_2026-06-14-132040.csv
Cleaned dataset saved: antenatal_clean_East Garo Hills_2026-04-05.csv
Summary recorded for district: East Garo Hills

🔹 Processing Antenatal file -> ML_East_Jaintia_Hills_Antenatal_care_2026-06-14-132407.csv
Cleaned dataset saved: antenatal_clean_East Jaintia Hills_2026-04-05.csv
Summary recorded for district: East Jaintia Hills

🔹 Processing Antenatal file -> ML_East_Khasi_Hills_Antenatal_care_2026-06-14-132232.csv
Cleaned dataset saved: antenatal_clean_East Khasi Hills_2026-04-05.csv
Summary recorded for district: East Khasi Hills

🔹 Processing Antenatal file -> ML_North_Garo_Hills_Antenatal_care

## Combining Script

### Antenatal

In [13]:
def reshape_antenatal(data, district_name):

    data['District'] = district_name

    assessment_counts = (
        data.groupby('Case ID')['Assessment date']
        .count()
        .rename('number_of_assessments')
    )
    data = data.merge(assessment_counts, on='Case ID')

    reshaped_data = data.sort_values(['Case ID', 'Assessment date']).copy()
    reshaped_data['assessment_number'] = reshaped_data.groupby('Case ID').cumcount() + 1

    static_columns = [
        'User Acc ID', 'User Name', 'NGO/Facility', 'Case ID', 'User Reg Id',
        'District', 'number_of_assessments', 'Taluka', 'Village', 'Awc Name No'
    ]
    static_columns = [c for c in static_columns if c in reshaped_data.columns]

    exclude_columns = {
        'Type of ANC Event', 'Pregnancy week during assessment',
        'Antenatal care stage', 'Select applicable options',
        'Fill details for "Other"', 'Mother\'s discomfort',
        'Mother\'s medical history', 'Mother\'s food intake',
        'Expected place for child delivery', 'Next Antenatal checkup date'
    }

    assessment_columns = [
        col for col in data.columns
        if col not in static_columns and col not in exclude_columns
    ]

    reshaped = reshaped_data.pivot(
        index=static_columns,
        columns='assessment_number',
        values=assessment_columns
    )

    flattened_columns = []
    max_assessments = reshaped.columns.levels[1].max()

    for num in range(1, max_assessments + 1):
        for col in assessment_columns:
            flattened_columns.append(f"{col}_{num}")

    reshaped.columns = [f"{col[0]}_{int(col[1])}" for col in reshaped.columns]
    reshaped = reshaped.reset_index()

    final_columns = static_columns + flattened_columns
    reshaped = reshaped[final_columns]

    return reshaped


### BF

In [14]:
import pandas as pd

def reshape_bf(df, district_name):

    # Step 1: Assign district
    df['District'] = district_name

    # Step 2: Number of assessments
    df['Assessment date'] = pd.to_datetime(df['Assessment date'], errors='coerce')
    assessment_counts = df.groupby('Case ID')['Assessment date'].count().rename('number_of_assessments')
    df = df.merge(assessment_counts, on='Case ID')

    # Step 3: Add assessment index
    reshaped_data = df.sort_values(['Case ID', 'Assessment date']).copy()
    reshaped_data['assessment_number'] = reshaped_data.groupby('Case ID').cumcount() + 1

    # STATIC columns
    static_columns = [
        'User Acc ID', 'User Name', 'NGO/Facility', 'Case ID', 'User Reg Id',
        'District', 'Taluka', 'Village', 'Awc Name No', 'number_of_assessments'
    ]
    static_columns = [c for c in static_columns if c in reshaped_data.columns]

    # ASSESSMENT columns -> everything except static
    assessment_columns = [
        col for col in df.columns
        if col not in static_columns
    ]

    # Pivoting
    reshaped = reshaped_data.pivot(
        index=static_columns,
        columns='assessment_number',
        values=assessment_columns
    )

    # Build expected flattened column list
    flattened_columns = []
    max_assessments = reshaped.columns.levels[1].max()

    for n in range(1, max_assessments + 1):
        for col in assessment_columns:
            flattened_columns.append(f"{col}_{n}")

    # Flatten MultiIndex -> colname_assessment#
    reshaped.columns = [f"{col}_{int(assess)}" for col, assess in reshaped.columns]
    reshaped = reshaped.reset_index()

    # Final ordering
    final_columns = static_columns + flattened_columns
    reshaped = reshaped[final_columns]

    return reshaped


### Case Measurement

In [15]:
import pandas as pd

# Function to process each dataset
def reshape_CM(data, district_name):
    # Step 1: Assign district name
    data['District'] = district_name

    # Step 2: Calculate the number of visits for each 'Case ID'
    data['Visit Date'] = pd.to_datetime(data['Visit Date'])  # Ensure dates are in datetime format
    visit_counts = data.groupby('Case ID')['Visit Date'].count().rename('number_of_visits')
    data = data.merge(visit_counts, on='Case ID')

    # Step 3: Reshape the dataset for visits
    reshaped_data = data.sort_values(['Case ID', 'Visit Date']).copy()
    reshaped_data['visit_number'] = reshaped_data.groupby('Case ID').cumcount() + 1

    # Static columns (not under assessment)
    static_columns = ['Reg ID', 'User Acc ID', 'User Name', 'Facility/NGO', 'User Role',
       'Case ID', 'Name of mother', 'Child ID', 'Child Name', 'Child Gender','District', 'number_of_visits']
    static_columns = [c for c in static_columns if c in reshaped_data.columns]

    # Assessment columns (all other columns except static ones)
    assessment_columns = [col for col in data.columns if col not in static_columns + ['visit_number']]

    # Pivot to create visit-specific columns for assessment columns
    reshaped = reshaped_data.pivot(index=static_columns, 
                                   columns='visit_number', 
                                   values=assessment_columns)

    # Rearrange the columns to group each visit's data together
    flattened_columns = []
    max_visits = reshaped.columns.levels[1].max()  # Get the maximum number of visits
    for visit_num in range(1, max_visits + 1):
        for col in assessment_columns:
            flattened_columns.append(f"{col}_{visit_num}")

    # Flatten and re-order columns based on the visit grouping
    reshaped.columns = [f"{col[0]}_{int(col[1])}" for col in reshaped.columns]
    reshaped = reshaped.reset_index()

    # Final column order, ensuring the static columns are at the front
    final_columns = static_columns + flattened_columns
    reshaped = reshaped[final_columns]

    return reshaped



### Case Report

In [16]:
import pandas as pd

def reshape_CR(data, district_name):
    # Add the district column
    data['District'] = district_name

    # Define the desired column order
    column_order = ['Reg ID', 'User Acc ID', 'User name', 'User Phone', 'User Role',
       'Mother Adoption date', 'Case ID', 'District','Name of mother', 'Pregnancy Stage',
       'Child ID', 'Child Name', 'Date of birth of baby', 'Gender of baby',
       'Birth Weight', 'Birth Height', 'Birth Weight Zscore',
       'Birth Height Zscore', 'Birth WFH Zscore', 'Baby Adoption date',
       'Baby Adoption Weight', 'Baby Adoption Height',
       'Baby Adoption Weight Zscore', 'Baby Adoption Height Zscore',
       'Baby Adoption WFH Zscore', 'Baby Adoption Percentile (W)',
       'Baby Adoption Percentile (H)', 'Baby Adoption Percentile (WFH)',
       'Last Visit Date', 'Last Weight', 'Last Height', 'Last Weight Zscore',
       'Last Height Zscore', 'Last WFH Zscore', 'Last Percentile (W)',
       'Last Percentile (H)', 'Last Percentile (WFH)', 'Phc Name',
       'Phc District', 'Phc Taluka', 'Member Tag', 'Cue Rating Tag',
       'Last Menstrual Period (LMP)', 'Estimated Date of Delivery (EDD)',
       'Case Rating', 'Comments', 'Viewed notification', 'Viewed planner']

    # Select only existing columns 
    data = data[[col for col in column_order if col in data.columns]]
    
    return data


### CF

In [17]:
import pandas as pd

# Function to process each dataset
def reshape_CF(data, district_name):
    # Step 1: Assign district name
    data['District'] = district_name

    # Step 2: Calculate the number of assessments for each 'Case ID'
    data['Assessment_date'] = pd.to_datetime(data['Assessment date'])  # Ensure dates are in datetime format
    assessment_counts = data.groupby('Case ID')['Assessment date'].count().rename('number_of_assessments')
    data = data.merge(assessment_counts, on='Case ID')

    # Step 3: Reshape the dataset for assessments
    reshaped_data = data.sort_values(['Case ID', 'Assessment date']).copy()
    reshaped_data['assessment_number'] = reshaped_data.groupby('Case ID').cumcount() + 1

    # Static columns (not under assessment)
    static_columns = ['User Acc ID', 'User Name', 'NGO/Facility', 'Case ID', 'User Reg Id', 'District','Taluka',
       'Village', 'Awc Name No', 'number_of_assessments']
    static_columns = [c for c in static_columns if c in reshaped_data.columns]
    
    # Exclusion list
    exclude_columns = {
        'Number of days per week whole beans or pulses were given to baby',
        'Select whole beans or pulses given to baby',
        'Did you give the baby whole beans or pulses in the last 24 hours?',
        'Number of days per week milk products were given to baby',
        'Select types of milk products given to baby',
        'Did you give the baby any milk items in the last 24 hours?',
        'Number of days per week grains were given to baby',
        'Select types of grains given to baby',
        'Did you give the baby any grains in the last 24 hours?',
        'Number of days per week millets were given to baby',
        'Select types of millets given to baby',
        'Did you give the baby any millets in the last 24 hours?',
        'Number of days per week green leafy vegetables were given to babt',
        'Did you give the baby any green leafy vegetables in the last 24 hours?',
        'Number of days per week red and orange vegetables were given to baby',
        'Did you give the baby any red and orange vegetables in the last 24 hours?',
        'Number of days per week other vegetables were given to baby',
        'Did you give the baby any other vegetables in the last 24 hours?',
        'Number of days per week fruits were given to baby',
        'Did you give the baby any fruits in the last 24 hours?',
        'Number of days per week roots and tubers were given to baby',
        'Did you give the baby any roots and tubers in the last 24 hours?',
        'Number of days per week nuts and seeds were given to baby',
        'Did you give the baby any nuts and seeds in the last 24 hours?',
        'Number of days per week eggs were given to baby',
        'Did you give the baby any eggs in the last 24 hours?',
        'Number of days per week chicken or poultry were given to baby',
        'Did you give the baby any chicken or poultry in the last 24 hours?',
        'Number of days per week seafood was given to baby',
        'Did you give the baby any seafood in the last 24 hours?',
        'Number of days per week meat and organs were given to baby',
        'Did you give the baby any meat or organs in the last 24 hours?',
        'Consistency of food given to the baby',
        'Number of meals per day',
        'Quantity of food given to the baby per meal',
        'Were the following ingredients added to the baby\'s food?',
        'Baby is breastfed or given expressed breast milk every day',
        'Type of water given to the baby',
        'Was a combination of cereals and pulses given?',
        'Which of the following was given to the baby?',
        'Is fruit puree added to the baby\'s regular meals?',
        'What cooking techniques were used to increase nutrient absorption for the food given to the baby?',
        'Which home-made nutritious powders was added to the food given to the baby'
    }

    # Assessment columns (all other columns except static and excluded ones)
    assessment_columns = [col for col in data.columns 
                          if col not in static_columns and col not in exclude_columns]

    # Pivot to create assessment-specific columns for assessment columns
    reshaped = reshaped_data.pivot(index=static_columns, 
                                   columns='assessment_number', 
                                   values=assessment_columns)

    # Rearrange the columns to group each assessment’s data
    flattened_columns = []
    max_assessments = reshaped.columns.levels[1].max()

    for assessment_num in range(1, max_assessments + 1):
        for col in assessment_columns:
            flattened_columns.append(f"{col}_{assessment_num}")

    # Flatten column names
    reshaped.columns = [f"{col[0]}_{int(col[1])}" for col in reshaped.columns]
    reshaped = reshaped.reset_index()

    # Final ordering
    final_columns = static_columns + flattened_columns
    reshaped = reshaped[final_columns]

    return reshaped



### CG

In [18]:
import pandas as pd

# Function to process each dataset for CG Step 2
def reshape_CG(data, district_name):
    # Step 1: Assign district name
    data['District'] = district_name

    # Step 2: Calculate the number of visits for each 'Case ID'
    data['Measurement date'] = pd.to_datetime(data['Measurement date'], errors="coerce")
    visit_counts = (
        data.groupby('Case ID')['Measurement date']
        .count()
        .rename('number_of_visits_CG')
    )
    data = data.merge(visit_counts, on='Case ID')

    # Step 3: Reshape dataset by visit
    reshaped_data = data.sort_values(['Case ID', 'Measurement date']).copy()
    reshaped_data['visit_number'] = (
        reshaped_data.groupby('Case ID').cumcount() + 1
    )

    # Columns that stay fixed
    static_columns = [
        'User Acc ID', 'User Name', 'NGO/Facility', 'Case ID', 'Child ID',
        'User Reg Id', 'Child Gender', 'District', 'Taluka', 'Village',
        'Awc Name No', 'number_of_visits_CG'
    ]
    static_columns = [c for c in static_columns if c in reshaped_data.columns]

    # All other columns except static + visit
    assessment_columns = [
        col for col in data.columns 
        if col not in static_columns + ['visit_number']
    ]

    # Pivot for visit-number columns
    reshaped = reshaped_data.pivot(
        index=static_columns,
        columns='visit_number',
        values=assessment_columns
    )

    # Flatten and reorder
    reshaped.columns = [f"{col[0]}_{int(col[1])}" for col in reshaped.columns]
    reshaped = reshaped.reset_index()

    # Final order
    max_visits = reshaped_data['visit_number'].max()
    flattened_columns = []
    for v in range(1, max_visits + 1):
        for col in assessment_columns:
            flattened_columns.append(f"{col}_{v}")

    final_columns = static_columns + flattened_columns
    reshaped = reshaped.reindex(columns=final_columns, fill_value=None)

    return reshaped


### Child

In [19]:
import pandas as pd

def reshape_child(data, district_name):
    # Add the district column
    data['District'] = district_name

    # Define the desired column order
    column_order = columns = [
    'User Acc ID', 
    'User Name', 
    'NGO/Facility', 
    'District', 
    'Submission_ID',
    'Submission_Date', 
    'Case ID', 
    'Babies born in this delivery',
    "Baby's name", 
    'Date of birth', 
    'Birth Weight (in Kgs)',
    'How was weight measured', 
    'Length at birth (in cms)',
    'Head circumference (in cms)', 
    "Baby's gender", 
    'Method of delivery',
    'Location of delivery', 
    'Select Primary Health Center (PHC)',
    'Select Sub Health Centre (SHC)',
    "Is this the mother's first pregnancy?", 
    "Number of child's siblings",
    'Was breast crawl performed at birth?',
    'Was baby exclusively breastfed within 1 hour of birth?',
    'Was baby exclusively breastfed during the ward stay?',
    'Food given to the baby at hospital', 
    'Type details of "other food"',
    'User Reg Id'
]
    
    # Select only columns available in dataset
    data = data[[col for col in column_order if col in data.columns]]
    
    return data


### Mother

In [20]:
import pandas as pd

def reshape_mother(data, district_name):
    # Add the district column
    data['District'] = district_name

    # Desired column order
    column_order = [
        'User Acc ID',
        'User Name',
        'NGO/Facility',
        'Submission_ID',
        'Submission_Date',
        'Case ID',
        'Role',
        'Adoption date',
        "Mother's Name",
        'Who is involved in this case?',
        'Last Menstrual Period (LMP)',
        'What was the delivery date?',
        'Stage at registration',
        'Current Month of Pregnancy',
        'Current month of Pregnancy',
        "Mother's Age (years)",
        "Mother's weight (kgs)",
        "Mother's height (cms)",
        'Diet Type',
        'Send notifications to mother?',
        "Mother's mobile number",
        'Email ID',
        "Mother's education level",
        'District or similar region',
        'Village or similar geographic area',
        'Block Name',
        "Enter if name of mother's village and block are different",
        "Color of mother's family ration card",
        # FIX (04-08-2026): this entry used to carry a TRAILING SPACE while the
        # normalised header has none, so the whitelist filter below silently dropped
        # the column and every ration-card "Other" explanation was lost.
        'Enter details for "Other" ration card',
        "Mother's family type",
        "Mother's social category",
        'Enter details for "Other"',
        'Estimated Date of Delivery (EDD)',
        'ID (cannot edit)',
        'User Reg Id',
        # FIX (04-08-2026): 'District' is set two lines above but was missing from this
        # whitelist, so it was immediately dropped again. Every other reshape_* keeps it.
        'District'
    ]

    # Keep only available columns.
    # FIX (04-08-2026): compare on the STRIPPED name in both directions, so a stray
    # leading/trailing space in either the whitelist or the export header can never
    # silently drop a column again (this is the second trailing-space data loss found
    # in this pipeline). Column names themselves are left exactly as they are.
    _avail = {}
    for _c in data.columns:
        _avail.setdefault(str(_c).strip(), _c)
    data = data[[_avail[_c.strip()] for _c in column_order if _c.strip() in _avail]]

    return data


### Protein

In [21]:
import pandas as pd

def reshape_protein(data, district_name):
    # Step 1: Assign district name
    data['District'] = district_name

    # Step 2: Calculate number of assessments
    data['Assessment date'] = pd.to_datetime(data['Assessment date'], errors='coerce')
    assessment_counts = (
        data.groupby('Case ID')['Assessment date']
        .count()
        .rename('number_of_assessments_protein')
    )
    data = data.merge(assessment_counts, on='Case ID')

    # Step 3: Create sorted visit sequence
    reshaped_data = data.sort_values(['Case ID', 'Assessment date']).copy()
    reshaped_data['visit_number'] = reshaped_data.groupby('Case ID').cumcount() + 1

    # Static columns
    static_columns = [
        'User Acc ID', 'User Name', 'NGO/Facility', 'Case ID', 
        'User Reg Id', 'District', 'number_of_assessments_protein'
    ]
    static_columns = [c for c in static_columns if c in reshaped_data.columns]

    # Assessment columns = everything else
    assessment_columns = [
        col for col in data.columns
        if col not in static_columns + ['visit_number']
    ]

    # Pivot: visit_number becomes wide columns
    reshaped = reshaped_data.pivot(
        index=static_columns,
        columns='visit_number',
        values=assessment_columns
    )

    # Build ordered flattened columns
    flattened_columns = []
    max_visits = reshaped.columns.levels[1].max()

    for visit_num in range(1, max_visits + 1):
        for col in assessment_columns:
            flattened_columns.append(f"{col}_{visit_num}")

    reshaped.columns = [f"{col[0]}_{int(col[1])}" for col in reshaped.columns]

    reshaped = reshaped.reset_index()
    reshaped = reshaped[static_columns + flattened_columns]

    return reshaped


### Running Combining Script

In [22]:
step2_config = {
    "antenatal": {
        "reshape": reshape_antenatal,
        "prefix": "antenatal_clean_"
    },
    "bf": {
        "reshape": reshape_bf,
        "prefix": "bf_clean_"
    },
    "case_measurement": {
        "reshape": reshape_CM,
        "prefix": "cm_clean_"
    },
    "case_report": {
        "reshape": reshape_CR,
        "prefix": "cr_clean_"
    },
    "cf": {
        "reshape": reshape_CF,
        "prefix": "cf_clean_"
    },
    "check_growth": {
        "reshape": reshape_CG,
        "prefix": "cg_clean_"
    },
    "child": {
        "reshape": reshape_child,
        "prefix": "child_clean_"
    },
    "mother": {
        "reshape": reshape_mother,
        "prefix": "mother_clean_"
    },
    "protein": {
        "reshape": reshape_protein,
        "prefix": "protein_clean_"
    }
}


In [23]:
def run_step2_generic(form_name, cleaned_folder, reshape_function, prefix, output_file):

    print(f"\n=== Running Step-2 for {form_name.upper()} ===")
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    processed = []

    for file in os.listdir(cleaned_folder):
        if not file.lower().startswith(prefix):
            continue
        if not file.lower().endswith(".csv"):
            continue

        print(f" -> Processing: {file}")

        parts = file.split("_")
        district = parts[2]

        df = pd.read_csv(os.path.join(cleaned_folder, file))
        reshaped = reshape_function(df, district)
        processed.append(reshaped)

    if not processed:
        print(f" No files found for {form_name}")
        return

    combined = pd.concat(processed, ignore_index=True)
    combined.to_csv(output_file, index=False)

    print(f"Completed: {output_file}")


In [24]:
def run_all_step2():

    for form_name, cfg in step2_config.items():

        cleaned_dir = f"cleaned_files/{form_name}/"
        output_file = f"combined_files/{form_name}_combined.csv"

        print(f"\n Running Step-2 for: {form_name}")

        run_step2_generic(
            form_name=form_name,
            cleaned_folder=cleaned_dir,
            reshape_function=cfg["reshape"],
            prefix=cfg["prefix"],
            output_file=output_file
        )

    print("\n All Step-2 processing completed successfully!")


In [25]:
if __name__ == "__main__":
    run_all_step2()



 Running Step-2 for: antenatal

=== Running Step-2 for ANTENATAL ===
 -> Processing: antenatal_clean_East Garo Hills_2026-04-05.csv
 -> Processing: antenatal_clean_East Jaintia Hills_2026-04-05.csv
 -> Processing: antenatal_clean_East Khasi Hills_2026-04-05.csv
 -> Processing: antenatal_clean_Eastern West Khasi Hills_2026-04-05.csv
 -> Processing: antenatal_clean_North Garo Hills_2026-04-05.csv
 -> Processing: antenatal_clean_Ri Bhoi_2026-04-05.csv
 -> Processing: antenatal_clean_South Garo Hills_2026-04-05.csv
 -> Processing: antenatal_clean_South West Garo Hills_2026-04-05.csv
 -> Processing: antenatal_clean_South West Khasi Hills_2026-04-05.csv
 -> Processing: antenatal_clean_West Garo Hills_2026-04-05.csv
 -> Processing: antenatal_clean_West Jaintia Hills_2026-04-05.csv
 -> Processing: antenatal_clean_West Khasi Hills_2026-04-05.csv
Completed: combined_files/antenatal_combined.csv

 Running Step-2 for: bf

=== Running Step-2 for BF ===
 -> Processing: bf_clean_East Garo Hills_2026

In [ ]:
# ==============================================================================
# PRE-MERGE DATA SCREENING ADD-ON (requested 04-08-2026, append-only)
# Runs on the RAW export files, BEFORE anything is merged into the master, and
# writes one shareable workbook that answers:
#   * for every variable of every form: how many rows are blank, and what % --
#     so a variable that is silently empty is visible before analysis starts;
#   * for the categorical variables of the Mother and Child forms: the full
#     frequency table (value, count, % of answered, % of all rows);
#   * for continuous variables: how many are non-blank, plus min/median/max so
#     impossible values stand out;
#   * how each form's Case IDs compare with the CASE LIST (the case-report
#     export) -- the "are the numbers we see in the case list actually backed by
#     data" check.
# Writes a NEW file only: summaries/data_screening_<dataset>.xlsx
# Nothing existing is read differently, changed or overwritten.
# Set GENERATE_DATA_SCREENING = False to switch the add-on off entirely.
# ==============================================================================
GENERATE_DATA_SCREENING = True

# Forms that get full frequency sheets (others still get blank-% in Overview)
SCREENING_FREQUENCY_FORMS = ["mother", "child"]
# A variable with more distinct values than this is treated as free text / an ID
SCREENING_MAX_CATEGORIES = 40
# Variables at or above this % blank are listed on the "Blank flags" sheet
SCREENING_BLANK_FLAG_PCT = 20.0

if GENERATE_DATA_SCREENING:

    print("\n" + "=" * 78)
    print("PRE-MERGE DATA SCREENING  (raw exports, before any merging)")
    print("=" * 78)

    # ---- which raw file belongs to which form (first match wins) -------------
    _SCR_FORM_RULES = [
        ("case_report",      lambda n: "case_report" in n),
        ("case_measurement", lambda n: "measure" in n),
        ("protein",          lambda n: "protein" in n),
        ("mother",           lambda n: "mother" in n),
        ("child",            lambda n: "child" in n),
        ("antenatal",        lambda n: "antenatal" in n),
        ("check_growth",     lambda n: "check_growth" in n),
        ("bf",               lambda n: "_bf" in n),
        ("cf",               lambda n: "_cf" in n),
    ]

    def _scr_form_of(fname):
        n = fname.lower()
        for form, rule in _SCR_FORM_RULES:
            if rule(n):
                return form
        return None

    def _scr_blank(series):
        """Blank = NaN/NaT, empty string, or a whitespace-only string."""
        s_na = series.isna()
        s_txt = series.astype(str).str.strip().str.lower()
        return s_na | s_txt.isin(["", "nan", "nat", "none", "null"])

    _SCR_DATEWORDS = ("date", "dob", "lmp", "edd", "timestamp", "submission_date")
    _SCR_TEXTWORDS = ("id", "name", "mobile", "email", "picture", "upload",
                      "remark", "comment", "details", "address", "phone")

    def _scr_kind(series, colname, n_nonblank, n_distinct):
        """Classify a variable so the right screening statistic is applied."""
        low = str(colname).lower()
        if any(w in low for w in _SCR_DATEWORDS):
            return "date"
        if n_nonblank == 0:
            return "empty"
        if any(w in low for w in _SCR_TEXTWORDS) and n_distinct > SCREENING_MAX_CATEGORIES:
            return "identifier/free-text"
        _num = pd.to_numeric(series, errors="coerce")
        if _num.notna().sum() >= 0.9 * n_nonblank:
            return "continuous" if n_distinct > 15 else "categorical"
        if n_distinct <= SCREENING_MAX_CATEGORIES:
            return "categorical"
        return "identifier/free-text"

    # ---- load every raw export ----------------------------------------------
    _scr_parts = {}
    for _f in sorted(os.listdir(RAW_INPUT_FOLDER)):
        if not _f.lower().endswith(".csv"):
            continue
        _form = _scr_form_of(_f)
        if _form is None:
            print(f"  [?] unrecognised raw file, skipped: {_f}")
            continue
        _scr_parts.setdefault(_form, []).append(
            (_f, pd.read_csv(os.path.join(RAW_INPUT_FOLDER, _f), low_memory=False)))

    # a state-level export can hold one file per district per form -> stack them
    _scr_raw = {}
    for _form, _items in _scr_parts.items():
        _names = ", ".join(_n for _n, _ in _items)
        _df_all = (_items[0][1] if len(_items) == 1
                   else pd.concat([_d for _, _d in _items], ignore_index=True))
        _scr_raw[_form] = (_names, _df_all)
        print(f"  read {_form:<17} {_df_all.shape[0]:>6} rows x "
              f"{_df_all.shape[1]:>3} cols   <- {_names}")

    # ---- 1) OVERVIEW: blanks per variable, every form ------------------------
    _scr_overview = []
    for _form, (_fname, _df_s) in _scr_raw.items():
        _n = len(_df_s)
        for _col in _df_s.columns:
            _s = _df_s[_col]
            _blank = int(_scr_blank(_s).sum())
            _nonblank = _n - _blank
            _distinct = int(_s[~_scr_blank(_s)].nunique())
            _kind = _scr_kind(_s, _col, _nonblank, _distinct)
            _row = {
                "Form": _form,
                "Variable": _col,
                "Type": _kind,
                "Rows": _n,
                "Non-blank": _nonblank,
                "Blank": _blank,
                "% blank": round(_blank / _n * 100, 2) if _n else 0.0,
                "Distinct values": _distinct,
                "Min": "", "Median": "", "Max": "", "Most common": "",
            }
            if _kind == "continuous":
                _num = pd.to_numeric(_s, errors="coerce")
                if _num.notna().any():
                    _row["Min"] = round(float(_num.min()), 2)
                    _row["Median"] = round(float(_num.median()), 2)
                    _row["Max"] = round(float(_num.max()), 2)
            elif _kind == "categorical" and _nonblank:
                _vc = _s[~_scr_blank(_s)].value_counts()
                _row["Most common"] = f"{_vc.index[0]} ({int(_vc.iloc[0])})"
            _scr_overview.append(_row)
    _scr_overview = pd.DataFrame(_scr_overview)

    # ---- 2) BLANK FLAGS: the screening shortlist -----------------------------
    _scr_flags = (_scr_overview[_scr_overview["% blank"] >= SCREENING_BLANK_FLAG_PCT]
                  .sort_values(["% blank", "Form", "Variable"], ascending=[False, True, True])
                  [["Form", "Variable", "Type", "Rows", "Non-blank", "Blank", "% blank"]])

    # ---- 3) COVERAGE vs the CASE LIST ---------------------------------------
    _scr_cov = []
    _case_ids = set()
    if "case_report" in _scr_raw and "Case ID" in _scr_raw["case_report"][1].columns:
        _cr_df = _scr_raw["case_report"][1]
        _case_ids = set(_cr_df.loc[~_scr_blank(_cr_df["Case ID"]), "Case ID"].astype(str).str.strip())
    for _form, (_fname, _df_s) in _scr_raw.items():
        _ids = set()
        if "Case ID" in _df_s.columns:
            _ids = set(_df_s.loc[~_scr_blank(_df_s["Case ID"]), "Case ID"].astype(str).str.strip())
        _scr_cov.append({
            "Form": _form,
            "Raw file": _fname,
            "Rows": len(_df_s),
            "Distinct Case IDs": len(_ids),
            "Case IDs also in case list": len(_ids & _case_ids),
            "Case IDs NOT in case list": len(_ids - _case_ids),
            "Case-list IDs with NO row in this form": len(_case_ids - _ids),
            "% of case list covered": (round(len(_ids & _case_ids) / len(_case_ids) * 100, 2)
                                       if _case_ids else 0.0),
        })
    _scr_cov = pd.DataFrame(_scr_cov)

    # ---- 4) FREQUENCY sheets for the requested forms -------------------------
    def _scr_freq_table(df_form):
        """Stacked frequency tables for every categorical variable of a form."""
        _out = []
        _n = len(df_form)
        for _col in df_form.columns:
            _s = df_form[_col]
            _bl = _scr_blank(_s)
            _nonblank = _n - int(_bl.sum())
            _distinct = int(_s[~_bl].nunique())
            if _scr_kind(_s, _col, _nonblank, _distinct) != "categorical":
                continue
            _out.append({"Variable": _col, "Value": "", "Count": "",
                         "% of answered": "", "% of all rows": ""})
            _vc = _s[~_bl].astype(str).str.strip().value_counts()
            for _val, _cnt in _vc.items():
                _out.append({
                    "Variable": "",
                    "Value": _val,
                    "Count": int(_cnt),
                    "% of answered": round(_cnt / _nonblank * 100, 2) if _nonblank else 0.0,
                    "% of all rows": round(_cnt / _n * 100, 2) if _n else 0.0,
                })
            _out.append({
                "Variable": "",
                "Value": "(blank / not answered)",
                "Count": int(_bl.sum()),
                "% of answered": "",
                "% of all rows": round(int(_bl.sum()) / _n * 100, 2) if _n else 0.0,
            })
            _out.append({
                "Variable": "", "Value": "TOTAL", "Count": _n,
                "% of answered": "", "% of all rows": 100.0,
            })
            _out.append({"Variable": "", "Value": "", "Count": "",
                         "% of answered": "", "% of all rows": ""})
        return pd.DataFrame(_out)

    # ---- 5) write the workbook ----------------------------------------------
    _scr_base = MERGED_OUTPUT_BASE if "MERGED_OUTPUT_BASE" in globals() else "dataset"
    _scr_name = f"data_screening_{_scr_base}.xlsx"
    _scr_path = os.path.join("summaries", _scr_name)
    os.makedirs("summaries", exist_ok=True)

    with pd.ExcelWriter(_scr_path, engine="openpyxl") as _w:
        _scr_cov.to_excel(_w, sheet_name="Coverage vs case list", index=False)
        _scr_flags.to_excel(_w, sheet_name="Blank flags", index=False)
        _scr_overview.to_excel(_w, sheet_name="Overview all variables", index=False)
        for _form in SCREENING_FREQUENCY_FORMS:
            if _form in _scr_raw:
                _scr_freq_table(_scr_raw[_form][1]).to_excel(
                    _w, sheet_name=f"Freq - {_form}"[:31], index=False)

    # ---- 6) console screening summary ---------------------------------------
    print("\nCoverage against the case list:")
    print(_scr_cov[["Form", "Rows", "Distinct Case IDs",
                    "Case IDs also in case list",
                    "Case IDs NOT in case list",
                    "% of case list covered"]].to_string(index=False))

    print(f"\nVariables at or above {SCREENING_BLANK_FLAG_PCT}% blank: {len(_scr_flags)}")
    if len(_scr_flags):
        print(_scr_flags.head(25).to_string(index=False))
        if len(_scr_flags) > 25:
            print(f"  ... and {len(_scr_flags) - 25} more (see the workbook)")

    for _form in SCREENING_FREQUENCY_FORMS:
        if _form not in _scr_raw:
            continue
        _sub = _scr_overview[_scr_overview["Form"] == _form]
        print(f"\n{_form.upper()} form: {len(_sub)} variables "
              f"({int((_sub['Type'] == 'categorical').sum())} categorical, "
              f"{int((_sub['Type'] == 'continuous').sum())} continuous, "
              f"{int((_sub['Type'] == 'empty').sum())} completely empty); "
              f"rows = {_scr_raw[_form][1].shape[0]}")
        _worst = _sub.sort_values("% blank", ascending=False).head(8)
        print(_worst[["Variable", "Type", "Non-blank", "Blank", "% blank"]].to_string(index=False))

    print(f"\nScreening workbook written -> {_scr_path}")
    print("=" * 78)


# Merging the Files Script

In [26]:

# -------------------------------------------------
# FORM -> SUFFIX MAPPING
# -------------------------------------------------
FORM_SUFFIX = {
    "case_measurement" : "",
    "case_report": "_CR",
    "bf": "_BF",
    "cf": "_CF",
    "child": "_C",
    "mother": "_M",
    "check_growth": "_CG",
    "protein": "_P",
    "antenatal": "_An"}

# FIX (04-08-2026): de-duplicate each form on Case ID BEFORE the outer merge and keep
# the MOST COMPLETE row (most non-blank cells) rather than whichever row happened to be
# first. Previously every form entered the merge with its duplicate Case IDs intact (an
# outer join multiplies them out) and a single drop_duplicates(keep="first") at the very
# end picked a row whose identity depended on os.listdir() ordering.
# Ties keep the original order, so a form whose first row was already the fullest is
# bit-for-bit unchanged -- verified against bf / cf / check_growth / protein, where
# keep="first" was already retaining the most complete row.
DEDUP_KEEP_MOST_COMPLETE = True

# -------------------------------------------------
# DETECT FORM NAME FROM FILENAME
# -------------------------------------------------
def detect_form_name(filename):
    name = filename.lower()
    for key in FORM_SUFFIX:
        if key in name:
            return key
    return None


# -------------------------------------------------
# APPLY SUFFIX TO COLUMN NAMES
# -------------------------------------------------
def add_suffix(df, suffix):
    new_cols = {}
    for col in df.columns:
        if col == "Case ID":      # Must not modify Case ID
            new_cols[col] = col
        else:
            new_cols[col] = f"{col}{suffix}"
    df = df.rename(columns=new_cols)
    return df


# ---------------------------------------
# MAIN MERGE LOGIC WITH FIXED ORDER
# ---------------------------------------
def run_clubbing_master(combined_folder="combined_files/", output_file="merged_files/ML_100526_V(120526).csv"):

    print("\n Starting master clubbing process...")

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    master_df = None  # Start empty

    # FIXED ORDER BASED ON FORM_SUFFIX KEYS
    ordered_forms = list(FORM_SUFFIX.keys())

    for form in ordered_forms:

        suffix = FORM_SUFFIX[form]

        # Find the file that contains this form name
        matching_files = [
            f for f in os.listdir(combined_folder)
            if f.lower().endswith(".csv") and form in f.lower()
        ]

        if not matching_files:
            print(f"No file found for form: {form}")
            continue

        # Usually only one file per form
        file = matching_files[0]
        file_path = os.path.join(combined_folder, file)

        print(f"➡ Reading: {file}   |   Form: {form}   |   Suffix: {suffix}")

        df = pd.read_csv(file_path)

        # Ensure Case ID exists
        if "Case ID" not in df.columns:
            print(f"ERROR: 'Case ID' missing in {file}. Skipping.")
            continue

        # Keep the most complete row per Case ID before merging (see
        # DEDUP_KEEP_MOST_COMPLETE above).
        if DEDUP_KEEP_MOST_COMPLETE and df["Case ID"].duplicated().any():
            _n_before = len(df)
            df = (df.assign(_completeness=df.notna().sum(axis=1))
                    .sort_values("_completeness", ascending=False, kind="mergesort")
                    .drop_duplicates(subset=["Case ID"], keep="first")
                    .drop(columns="_completeness")
                    .sort_index())
            print(f"   de-duplicated on Case ID: {_n_before} -> {len(df)} rows "
                  f"(kept the most complete row per case)")

        # Add suffix to all form columns
        df = add_suffix(df, suffix)

        # Merge with master respecting fixed form order
        if master_df is None:
            master_df = df
        else:
            master_df = master_df.merge(df, on="Case ID", how="outer")

    # -----------------------------
    # REMOVE DUPLICATES
    # -----------------------------
    if master_df is None:
        print("No valid files found - master file not created.")
        return

    master_df = master_df.drop_duplicates(subset=["Case ID"], keep="first")

    # Export final file
    master_df.to_csv(output_file, index=False)

    print(f"\n Master file successfully created:")
    print(f" {output_file}\n")

# -------------------------------------------------
# RUN
# -------------------------------------------------
if __name__ == "__main__":
    # Dynamic output naming: the merged master file is named after the raw
    # input folder (e.g. UJ_110726_V(170726).csv); original name as fallback.
    _merged_name = (MERGED_OUTPUT_BASE
                    if 'MERGED_OUTPUT_BASE' in globals() and MERGED_OUTPUT_BASE
                    else "ML_100526_V(120526)")
    run_clubbing_master(output_file="merged_files/" + _merged_name + ".csv")



 Starting master clubbing process...
➡ Reading: case_measurement_combined.csv   |   Form: case_measurement   |   Suffix: 
➡ Reading: case_report_combined.csv   |   Form: case_report   |   Suffix: _CR
➡ Reading: bf_combined.csv   |   Form: bf   |   Suffix: _BF
➡ Reading: cf_combined.csv   |   Form: cf   |   Suffix: _CF
➡ Reading: child_combined.csv   |   Form: child   |   Suffix: _C
➡ Reading: mother_combined.csv   |   Form: mother   |   Suffix: _M
➡ Reading: check_growth_combined.csv   |   Form: check_growth   |   Suffix: _CG
➡ Reading: protein_combined.csv   |   Form: protein   |   Suffix: _P
➡ Reading: antenatal_combined.csv   |   Form: antenatal   |   Suffix: _An

 Master file successfully created:
 merged_files/ML_100526_V(120526).csv



## 